In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, classification_report,
    mean_squared_error, r2_score
)

In [2]:
import os
os.makedirs("model", exist_ok=True)
os.makedirs("data", exist_ok=True)

In [7]:
# ---------------------------------------------------------------------------
# 1. MODELE 1 : RECOMMANDATION DE CULTURE (classification)
# ---------------------------------------------------------------------------
def train_crop_recommendation_model():
    print("=" * 60)
    print("MODELE 1 : Recommandation de culture")
    print("=" * 60)

    df = pd.read_csv(r"F:\data\Crop_recommendation.csv")
    print(f"Shape : {df.shape}")
    print(f"Cultures uniques ({df['label'].nunique()}) : {sorted(df['label'].unique())}")

    # Variables explicatives (X) et cible (y)
    X = df[["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]]
    y = df["label"]

    # Split train/test : 80/20, stratifié pour garder les proportions de chaque culture
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # RandomForest : robuste, pas besoin de normaliser les features, gère bien
    # le mélange d'échelles (N/P/K en unités, ph entre 0-14, rainfall en mm...)
    model = RandomForestClassifier(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    print(f"\nAccuracy : {accuracy_score(y_test, y_pred):.4f}")
    print("\nRapport de classification :")
    print(classification_report(y_test, y_pred))

    # Importance des variables (utile pour l'interprétation agronomique)
    importances = pd.Series(model.feature_importances_, index=X.columns)
    print("\nImportance des variables :")
    print(importances.sort_values(ascending=False))

    # Sauvegarde
    with open("model/crop_recommendation_model.pkl", "wb") as f:
        pickle.dump(model, f)
    print("\n-> Sauvegardé : model/crop_recommendation_model.pkl")

    return model


# ---------------------------------------------------------------------------
# 2. MODELE 2 : PREDICTION DE RENDEMENT (régression)
# ---------------------------------------------------------------------------
def train_yield_prediction_model():
    print("\n" + "=" * 60)
    print("MODELE 2 : Prédiction de rendement")
    print("=" * 60)

    df = pd.read_csv(r"F:\data\crop_yield.csv")
    df = df.sample(n=100_000, random_state=42)  
    print(f"Shape : {df.shape}")
    print(f"Cultures uniques : {sorted(df['Crop'].unique())}")

    # Colonnes catégorielles à encoder
    categorical_cols = ["Region", "Soil_Type", "Crop", "Weather_Condition"]
    # Colonnes booléennes déjà en True/False -> converties en 0/1
    bool_cols = ["Fertilizer_Used", "Irrigation_Used"]

    encoders = {}
    df_encoded = df.copy()

    for col in categorical_cols:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col])
        encoders[col] = le  # on garde chaque encodeur pour pouvoir décoder/encoder plus tard dans l'app

    for col in bool_cols:
        df_encoded[col] = df_encoded[col].astype(int)

    feature_cols = [
        "Region", "Soil_Type", "Crop", "Rainfall_mm", "Temperature_Celsius",
        "Fertilizer_Used", "Irrigation_Used", "Weather_Condition", "Days_to_Harvest"
    ]
    X = df_encoded[feature_cols]
    y = df_encoded["Yield_tons_per_hectare"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor(
    n_estimators=200,      # 300 -> 200, suffisant sur un aussi gros dataset
    max_depth=15,          # limite la profondeur, accélère beaucoup
    n_jobs=-1,             # utilise tous les cœurs du CPU (gain énorme)
    random_state=42
)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"\nRMSE : {rmse:.4f} tonnes/hectare")
    print(f"R²   : {r2:.4f}")

    importances = pd.Series(model.feature_importances_, index=feature_cols)
    print("\nImportance des variables :")
    print(importances.sort_values(ascending=False))

    # Sauvegarde du modèle + des encodeurs (indispensables pour encoder
    # les nouvelles entrées de la même façon dans l'app Streamlit)
    with open("model/yield_prediction_model.pkl", "wb") as f:
        pickle.dump(model, f)
    with open("model/label_encoders.pkl", "wb") as f:
        pickle.dump(encoders, f)
    print("\n-> Sauvegardé : model/yield_prediction_model.pkl")
    print("-> Sauvegardé : model/label_encoders.pkl")

    return model, encoders


# ---------------------------------------------------------------------------
if __name__ == "__main__":
    train_crop_recommendation_model()
    train_yield_prediction_model()
    print("\nEntraînement terminé. Les fichiers .pkl sont prêts pour app.py")


MODELE 1 : Recommandation de culture
Shape : (2200, 8)
Cultures uniques (22) : ['apple', 'banana', 'blackgram', 'chickpea', 'coconut', 'coffee', 'cotton', 'grapes', 'jute', 'kidneybeans', 'lentil', 'maize', 'mango', 'mothbeans', 'mungbean', 'muskmelon', 'orange', 'papaya', 'pigeonpeas', 'pomegranate', 'rice', 'watermelon']

Accuracy : 0.9955

Rapport de classification :
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      0.95      0.97        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.95      1.00      0.98        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       1.00      1.0

In [9]:
import os
print(os.getcwd())

c:\Users\PC Paradise\AppData\Local\Programs\Microsoft VS Code


In [13]:
 import os
 os.chdir(r"F:\crop project")
 print(os.getcwd())  # vérifie que ça a bien changé

F:\crop project


In [14]:
%run train_models.py

Exception: File `'train_models.py'` not found.